# Spark RDD Word Count with HDFS

This notebook uses Apache Spark 3.5.9 to read a text file from HDFS and count words with RDD transformations and actions.

You will create a small local text file, upload it to HDFS, inspect input partitions, clean the text, count the words, and save the result to HDFS.

## 1. Start the required services

Start HDFS and the Spark standalone master and worker before opening this notebook:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

Confirm that `NameNode`, `DataNode`, `SecondaryNameNode`, `Master`, and `Worker` appear. This notebook does not use YARN.

## 2. Check HDFS

The report must show one live DataNode.

In [ ]:
%%bash
hdfs dfsadmin -safemode get
hdfs dfsadmin -report | grep -E 'Live datanodes|Dead datanodes'

## 3. Create a small local input file

The file contains uppercase letters, periods, commas, a colon, and an exclamation mark. We will remove this punctuation later with Spark.

In [ ]:
%%bash
printf '%s\n' \
  'Spark reads data. Spark processes data!' \
  'RDD transformations are lazy.' \
  'Actions start Spark jobs, and tasks process partitions.' \
  'Data, data, and more DATA: this is a word-count example.' \
  > /tmp/d281_sample.txt

cat /tmp/d281_sample.txt

## 4. Upload the file to HDFS

Each student writes below their own HDFS home directory because the path uses `$USER`. `-put -f` replaces the file when the cell is run again.

In [ ]:
%%bash
HDFS_INPUT="/user/$USER/d281/input"

hdfs dfs -mkdir -p "$HDFS_INPUT"
hdfs dfs -put -f /tmp/d281_sample.txt "$HDFS_INPUT/sample.txt"
hdfs dfs -ls -h "$HDFS_INPUT"
hdfs dfs -cat "$HDFS_INPUT/sample.txt"

## 5. Connect to the Spark standalone master

`HADOOP_CONF_DIR` must already point to the Hadoop configuration directory. Spark uses that configuration to locate HDFS.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D281-Spark-WordCount")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 6. Build the HDFS paths

The Python username must match `$USER` from the shell cells. An absolute path beginning with `hdfs:///` uses the NameNode configured by `fs.defaultFS`.

In [ ]:
hdfs_user = os.environ["USER"]
hdfs_input = f"hdfs:///user/{hdfs_user}/d281/input/sample.txt"
hdfs_output = f"hdfs:///user/{hdfs_user}/d281/output/wordcount"

print("Input :", hdfs_input)
print("Output:", hdfs_output)

## 7. Read the HDFS text file

`sc.textFile` creates an RDD in which each record is one line of text. The call is lazy: Spark records the input dependency but does not read all lines until an action runs.

In [ ]:
lines_rdd = sc.textFile(hdfs_input)

print("RDD created:", lines_rdd)

## 8. Inspect the input partitions

`getNumPartitions()` reports how many RDD partitions Spark created. It does not start a job. `glom().collect()` starts a job and shows the lines held by every partition.

In [ ]:
total_partitions = lines_rdd.getNumPartitions()

print("Total partitions:", total_partitions)

In [ ]:
sc.setJobGroup("inspect-input", "Inspect text input partitions")

for partition_id, lines in enumerate(lines_rdd.glom().collect()):
    print(f"Partition {partition_id}: {lines}")

### How `textFile` partitions the input

For HDFS input, Spark uses Hadoop input splits. Splits are influenced mainly by file boundaries, file size, and HDFS block size. A tiny file commonly produces one input partition. A large splittable text file can produce several partitions, often near its HDFS block boundaries. A directory containing multiple files can also produce multiple partitions.

Text records are lines. If a split boundary falls in the middle of a line, the reader handles the boundary so the complete line is returned once. Some compressed formats are not splittable and may remain one partition per file.

`sc.textFile(path, minPartitions=4)` can request a minimum number of partitions, but it is a hint rather than a guarantee. More partitions create more tasks; partitions that are too small add scheduling overhead.

## 9. Convert every line to lowercase

`map` applies the function to every input record. This transformation is lazy.

In [ ]:
lowercase_rdd = lines_rdd.map(lambda line: line.lower())

Run an action to inspect the lowercase lines.

In [ ]:
lowercase_rdd.collect()

## 10. Remove periods and other special characters

The regular expression replaces every character that is not a lowercase letter, digit, or whitespace with a space. This removes periods and the other punctuation in the sample.

In [ ]:
import re

clean_lines_rdd = lowercase_rdd.map(
    lambda line: re.sub(r"[^a-z0-9\s]", " ", line)
)

In [ ]:
clean_lines_rdd.collect()

## 11. Split lines into words

`flatMap` can produce several output records from one input line. `split()` treats repeated whitespace as one separator.

In [ ]:
words_rdd = clean_lines_rdd.flatMap(lambda line: line.split())

In [ ]:
words_rdd.collect()

## 12. Create `(word, 1)` pairs

Each word becomes a key paired with the number 1.

In [ ]:
word_pairs_rdd = words_rdd.map(lambda word: (word, 1))

In [ ]:
word_pairs_rdd.take(10)

## 13. Count each word

`reduceByKey` adds the values for matching words. It requires a shuffle because copies of the same word may exist in different input partitions. The shuffle creates a new stage.

In [ ]:
word_counts_rdd = word_pairs_rdd.reduceByKey(
    lambda left_count, right_count: left_count + right_count
)

## 14. Sort by count and show the most frequent words

The sort key uses negative counts for descending order and the word for a stable alphabetical tie-break. `take(10)` is the action that starts the job.

In [ ]:
sorted_counts_rdd = word_counts_rdd.sortBy(
    lambda item: (-item[1], item[0])
)

In [ ]:
sc.setJobGroup("top-words", "Find the ten most frequent words")

top_words = sorted_counts_rdd.take(10)
for word, count in top_words:
    print(f"{word:<15} {count}")

## 15. Find the occurrence count of one word

Change `search_word` to look for another word. `lookup` returns the values belonging to that key.

In [ ]:
search_word = "spark"
matches = word_counts_rdd.lookup(search_word.lower())
occurrences = matches[0] if matches else 0

print(f"'{search_word}' occurs {occurrences} time(s).")

## 16. Inspect the lineage

The lineage shows the transformations used to create the sorted word counts. Look for shuffle dependencies introduced by `reduceByKey` and `sortBy`.

In [ ]:
print(sorted_counts_rdd.toDebugString().decode("utf-8"))

## 17. Delete the old HDFS output directory

Spark refuses to overwrite an existing output directory. Delete only this lesson's previous output before saving again.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/d281/output/wordcount"

hdfs dfs -rm -r -f "$HDFS_OUTPUT"

## 18. Save the word counts to HDFS

`saveAsTextFile` is an action. Spark creates the output directory and writes one `part-*` file per output partition.

In [ ]:
sc.setJobGroup("save-word-counts", "Save word counts to HDFS")
sorted_counts_rdd.saveAsTextFile(hdfs_output)

print("Saved to:", hdfs_output)

## 19. Verify the HDFS result

The output is stored as Python tuple text because the RDD contains `(word, count)` tuples.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/d281/output/wordcount"

hdfs dfs -ls -h "$HDFS_OUTPUT"
hdfs dfs -cat "$HDFS_OUTPUT"/part-*

## 20. Jobs, stages, partitions, and tasks

Open the Spark application UI printed earlier, normally `http://localhost:4040`.

- An **action** starts a job. Examples here include `collect`, `take`, `lookup`, and `saveAsTextFile`.
- A job is divided into **stages** at shuffle boundaries.
- `reduceByKey` and `sortBy` introduce shuffles.
- A stage normally runs one **task** for each partition in that stage.
- The number of input partitions can differ from the number of later shuffle partitions.

Select a job, open each stage, and compare **Number of Tasks** with the RDD partition counts.

## 21. Student exercise: Shakespeare books

Download one or more public-domain Shakespeare books from a trusted source and upload the text files to your D281 HDFS input directory. Keep the original files in `/tmp` only as temporary local copies.

Then adapt this notebook by changing `hdfs_input` to the uploaded file or input directory. The same RDD pipeline can process one book or several text files.

Complete these tasks:

1. Print the number of input partitions.
2. Convert all text to lowercase and remove punctuation.
3. Count every word.
4. Display the 20 most frequent words.
5. Find the occurrence count of a chosen word such as `king`, `love`, or `romeo`.
6. Compare that chosen word across two books by processing each book separately.
7. Inspect the jobs, stages, and task counts in the Spark UI.
8. Save the final counts to a new HDFS output directory.

Large books contain common words such as `the` and `and`. As an extension, remove common stop words before finding the most frequent words.

## 22. Stop Spark

Run this cell when the notebook is complete.

In [ ]:
spark.stop()
print("Spark session stopped.")